# P0 · Hybrid memory: visual words + real patches (exploratory)

One memory bank of M vectors: **half visual words** (k-means averages, which cover the common middle of
normal appearance) and **half real patches**, picked greedily as the normal patches the words cover *worst*
(rare-but-normal details). A test patch is scored by its distance to the nearest item of either kind.

Compared at the same M (64, 256, 1,024) with the codebook and coreset results already on Drive.
Reuses cached features: **no dataset download**. Runtime (T4): about 1.5–2 h for MVTec AD + VisA, mostly k-means.
Exploratory (designed after seeing these datasets); a confirmatory test should be preregistered.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Run (MVTec AD, then VisA)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
from bovw.sweep import load_config
from bovw import anomaly

results = {}
for ds in ["mvtec", "visa"]:
    cfg = load_config(f"configs/p0_hybrid_{ds}.yaml")
    results[ds] = anomaly.run(cfg)

## 3 · Compare with codebook and coreset
Means over categories and 3 seeds; `hybrid_w50` = 50% words, 50% real patches.

In [ ]:
import pandas as pd
RES = "/content/drive/MyDrive/bovw-forensics/results"
base = {"mvtec": "p0_anomaly_mvtec_v2.csv", "visa": "p0_anomaly_visa.csv"}
tables = {ds: anomaly.hybrid_compare(pd.read_csv(f"{RES}/{base[ds]}"), results[ds]) for ds in results}
out = pd.concat(tables, names=["dataset"])
out.to_csv(f"{RES}/p0_hybrid_compare.csv")
out.round(3)